# 03 — TF-IDF + Logistic Regression Baseline

Train a traditional text-classification baseline. Fit the vectorizer on training text only. Use validation results for development; keep test data untouched until notebook 06.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import json
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from src.config import PROCESSED_DIR, MODEL_DIR, METRICS_DIR, CLASS_NAMES, RANDOM_SEED
from src.evaluate import classification_metrics

MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "validation.csv")


In [ ]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=50_000, min_df=2, sublinear_tf=True)),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_SEED)),
])
baseline.fit(train_df["review_clean"].astype(str), train_df["label_id"].astype(int))
val_pred = baseline.predict(val_df["review_clean"].astype(str))
val_metrics = classification_metrics(val_df["label_id"].astype(int), val_pred)
print(json.dumps(val_metrics, indent=2))
print(classification_report(
    val_df["label_id"].astype(int), val_pred,
    labels=[0, 1, 2], target_names=CLASS_NAMES, zero_division=0
))
joblib.dump(baseline, MODEL_DIR / "tfidf_logreg.joblib")
(METRICS_DIR / "tfidf_validation_metrics.json").write_text(
    json.dumps(val_metrics, indent=2), encoding="utf-8"
)


## Notes

Record vectorizer settings, validation performance, and any warning. Do not select or tune the baseline using the held-out test set.